In [18]:
import numpy as np
from helpers import load_model, save_model, load_dataset
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [19]:
# 1. Load dataset
X, y = load_dataset("data_students.txt")
X = np.asarray(X)
y = np.asarray(y)
print("Dataset Shape:")
print(X.shape)
print(y.shape)

Dataset Shape:
(227978, 5)
(227978,)


In [20]:
# 2. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [24]:
# 3. Create Final Model
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

In [25]:
# 4. Train
model.fit(X, y)
print("Training finished.")

Training finished.


In [26]:
from sklearn.metrics import r2_score
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(f"R² = {r2:.6f}")

R² = 0.998217


In [27]:
save_model(model, "Compress_Model.pkl")
print("Model saved.")

Model saved.


In [33]:
import numpy as np
import pandas as pd
import tempfile
import pickle
import os

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score

# 搜索空间（建议缩小）
n_estimators_list = [20, 30, 40]
max_depth_list = [None, 16, 18, 20]
min_samples_leaf_list = [1, 2, 3]
max_features_list = [1.0, "sqrt"]

# 控制参数（关键加速）
CV = 3
SIZE_LIMIT_MB = 50
EARLY_STOP_THRESHOLD = 0.002  # 如果比当前best差0.002就跳过
results = []
best_cv = -np.inf

# 开始搜索
for n_estimators in n_estimators_list:
    for max_depth in max_depth_list:
        for min_leaf in min_samples_leaf_list:
            for max_features in max_features_list:

                # 1. 结构性早停（模型太大直接跳）
                if max_depth is not None:
                    if n_estimators * max_depth > 400:
                        continue

                model = RandomForestRegressor(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_leaf=min_leaf,
                    max_features=max_features,
                    random_state=42,
                    n_jobs=-1
                )

                # 2. 3-fold CV 评分
                cv_scores = cross_val_score(
                    model,
                    X_train,
                    y_train,
                    scoring="r2",
                    cv=CV,
                    n_jobs=-1
                )

                cv_mean = cv_scores.mean()

                # 3. early stop（核心）
                if cv_mean < best_cv - EARLY_STOP_THRESHOLD:
                    continue

                # 更新 best
                if cv_mean > best_cv:
                    best_cv = cv_mean

                # 4. 训练最终模型
                model.fit(X_train, y_train)
                pred = model.predict(X_test)
                r2 = r2_score(y_test, pred)

                # 5. 模型大小估计
                with tempfile.NamedTemporaryFile(delete=False) as f:
                    pickle.dump(model, f, protocol=4)
                    temp_name = f.name

                size_mb = os.path.getsize(temp_name) / 1024 / 1024
                os.remove(temp_name)

                # 6. size filter
                if size_mb > SIZE_LIMIT_MB:
                    continue

                results.append({
                    "n_estimators": n_estimators,
                    "max_depth": max_depth,
                    "min_samples_leaf": min_leaf,
                    "max_features": max_features,
                    "CV_R2": cv_mean,
                    "Test_R2": r2,
                    "File_Size_MB": size_mb
                })

                print(
                    f"CV={cv_mean:.5f}, "
                    f"Test={r2:.5f}, "
                    f"Size={size_mb:.1f}MB"
                )

# 汇总结果

df = pd.DataFrame(results)
df = df.sort_values(
    by=["CV_R2", "Test_R2", "File_Size_MB"],
    ascending=[False, False, True]
)
print(df)
df.to_csv("optimized_rf_search.csv", index=False)

KeyError: 'CV_R2'